In [30]:
%pip install -q onnxruntime


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import chromadb

In [2]:

import pandas as pd
from ydata_profiling import ProfileReport
import datamart_profiler
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import glob
import json
import sys
sys.path.append(r'c:\Users\ricca\Documents\Polimi\tesi\code')
import utils
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm  # For progress bar
from multiprocessing import Pool

import importlib
importlib.reload(utils)



<module 'utils' from 'c:\\Users\\ricca\\Documents\\Polimi\\tesi\\code\\utils.py'>

In [3]:
chroma_client = chromadb.PersistentClient()
embedding_function = utils.OpenAIEmbeddingFunction()
collection = chroma_client.get_collection(
    name="instructions_benchmark",
    embedding_function=embedding_function
)

In [4]:
def query_to_vector_db(query_text, n):
    results = collection.query(
        query_texts=[query_text],
        n_results=n
    )

    for i, doc in enumerate(results['documents'][0]):
        print(f"{i+1}. {doc} (ID: {results['ids'][0][i]}, Distance: {results['distances'][0][i]:.4f})")

    return results

In [5]:
import pickle

with open('instructions_dict.pkl', 'rb') as f:
    instructions_dict = pickle.load(f)

print(instructions_dict)

[{'unique_code': 'DS1-E-0005_Q01', 'query_id': 'DS1-E-0005', 'instruction': '1. "Latest global birth rates by month 2023'}, {'unique_code': 'DS1-E-0005_Q02', 'query_id': 'DS1-E-0005', 'instruction': '2. "Seasonal birth patterns in the United States CDC 2023 data'}, {'unique_code': 'DS1-E-0005_Q03', 'query_id': 'DS1-E-0005', 'instruction': '3. "Impact of economic recessions on birth rates recent studies'}, {'unique_code': 'DS1-E-0005_Q04', 'query_id': 'DS1-E-0005', 'instruction': '4. "Cultural practices influencing birth rates in South India 2023'}, {'unique_code': 'DS1-E-0005_Q05', 'query_id': 'DS1-E-0005', 'instruction': '5. "Changes in public health policies affecting birth rates 2023'}, {'unique_code': 'DS1-E-0005_Q06', 'query_id': 'DS1-E-0005', 'instruction': '6. "Effect of natural disasters on conception rates 2023'}, {'unique_code': 'DS1-E-0005_Q07', 'query_id': 'DS1-E-0005', 'instruction': '7. "Trends in assisted reproductive technologies and monthly birth distribution'}, {'uniq

In [6]:
from collections import defaultdict

# Raggruppa le instructions per query_id
instructions_by_query = defaultdict(list)
for item in instructions_dict:
    instructions_by_query[item['query_id']].append(item['instruction'])

# Effettua una query al vector db per ogni instruction di ogni query_id
all_retrieval_results = []

for query_id, instructions in instructions_by_query.items():
    for instruction in instructions:
        result = collection.query(
            query_texts=[instruction],
            n_results=5
        )
        all_retrieval_results.append({
            'query_id': query_id,
            'query_text': instruction,
            'instruction': instruction,
            'results': result
        })

# Visualizza i risultati per ogni instruction
for entry in all_retrieval_results:
    print(f"Query ID: {entry['query_id']}")
    print(f"Instruction: {entry['instruction']}")
    for i, doc in enumerate(entry['results']['documents'][0]):
        print(f"  {i+1}. {doc} (ID: {entry['results']['ids'][0][i]}, Distance: {entry['results']['distances'][0][i]:.4f})")
    print('-' * 60)

Query ID: DS1-E-0005
Instruction: 1. "Latest global birth rates by month 2023
  1. Show me the total number of live births and deaths in the United States for each month of 2023. (ID: instruction_17_benchmark/datasets_90_csv\8fb87dcb037b78f563ffd1bb1f66796264ef183c46ed5ca6d5fe721472a6a4e8.text.csv, Distance: 0.8382)
  2. Show me data on U.S. birth rates from 1990 to 2019. (ID: instruction_1_benchmark/datasets_90_csv\87e7971bdab71e198a058ffd70ab0712c75bc7fd0a4a35863e14b0c6280a01af.text.csv, Distance: 0.8522)
  3. What datasets provide birth rate information aggregated by year and age group? (ID: instruction_2_benchmark/datasets_90_csv\9685daa09bef973588b5113ee267256a2505f7a3387c4c6bcd54ca563b1b2395.text.csv, Distance: 0.8951)
  4. Search for a dataset that tracks birth rates by age group from 1940 to 2018. (ID: instruction_10_benchmark/datasets_90_csv\77f1f600691c07c7eb8c1b2205b674d88c23fd0911e9c1a41ae3cae7116a3769.text.csv, Distance: 0.9036)
  5. Show birth rate trends from 1970 to 201

In [7]:
def extract_id_from_path(path):
    # Estrae la parte tra l'ultimo '\' e '.text.csv'
    return path.split('\\')[-1].replace('.text.csv', '')

In [8]:
dataset_ids_by_query = {}

for entry in all_retrieval_results:
    qid = entry['query_id']
    if qid not in dataset_ids_by_query:
        dataset_ids_by_query[qid] = []
    for metadata in entry['results']['metadatas'][0]:
        dataset_id = metadata.get('dataset')
        if dataset_id and dataset_id not in dataset_ids_by_query[qid]:
            dataset_ids_by_query[qid].append(dataset_id)

print(dataset_ids_by_query)

{'DS1-E-0005': ['benchmark/datasets_90_csv\\8fb87dcb037b78f563ffd1bb1f66796264ef183c46ed5ca6d5fe721472a6a4e8.text.csv', 'benchmark/datasets_90_csv\\87e7971bdab71e198a058ffd70ab0712c75bc7fd0a4a35863e14b0c6280a01af.text.csv', 'benchmark/datasets_90_csv\\9685daa09bef973588b5113ee267256a2505f7a3387c4c6bcd54ca563b1b2395.text.csv', 'benchmark/datasets_90_csv\\77f1f600691c07c7eb8c1b2205b674d88c23fd0911e9c1a41ae3cae7116a3769.text.csv', 'benchmark/datasets_90_csv\\89fc43a2a9145df88d15f3fd27be6b917988d39e8420d8f0222936e04457c1ab.text.csv', 'benchmark/datasets_90_csv\\574f1eb54afbd37f07bce149907cb95ef50004c13ad0630654a1a5ac05355813.text.csv', 'benchmark/datasets_90_csv\\272911234aba536410e9fb0d0ed5830b10c56a1e6edc3f36600a49c380c12df6.text.csv', 'benchmark/datasets_90_csv\\73e052e31264f32135dda1f7c7eb21e1d1c9d326ce4d6cbb335476eb643cad13.text.csv', 'benchmark/datasets_90_csv\\998febc9d9d5b24f028cbc56bd7b26b9bd37fc3b23de509574150cff61e0c900.text.csv', 'benchmark/datasets_90_csv\\fdecb95234b4335cd1f7

In [9]:
import json

dataset_ids_from_jsonl = {}

with open('benchmark/data_search_e_collection.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        if 'data' in record and isinstance(record['data'], list):
            for data_entry in record['data']:
                dataset_name = data_entry.get('data_filename')
                if dataset_name:
                    for qid, dataset_list in dataset_ids_by_query.items():
                        for ds in dataset_list:
                            # Confronta solo il nome del file, non il path completo
                            if os.path.basename(dataset_name) == os.path.basename(ds):
                                if qid not in dataset_ids_from_jsonl:
                                    dataset_ids_from_jsonl[qid] = {}
                                dataset_ids_from_jsonl[qid][ds] = record.get('id')

print(dataset_ids_from_jsonl)

{'DS1-E-0005': {'benchmark/datasets_90_csv\\4583e0c59aba1daa2b75df1559e23dc0f12048847577f17d482690e37eba415d.text.csv': '0cbb09dc-4012-4f18-900e-6d746bfc10d3', 'benchmark/datasets_90_csv\\574f1eb54afbd37f07bce149907cb95ef50004c13ad0630654a1a5ac05355813.text.csv': '1560c507-f481-4f83-a9e1-94f42cb34aab', 'benchmark/datasets_90_csv\\9685daa09bef973588b5113ee267256a2505f7a3387c4c6bcd54ca563b1b2395.text.csv': '2a3cc123-f9f6-4e1d-a03f-3e33056e5021', 'benchmark/datasets_90_csv\\77f1f600691c07c7eb8c1b2205b674d88c23fd0911e9c1a41ae3cae7116a3769.text.csv': '579261e5-5c44-44c5-a948-8c47c9e63f66', 'benchmark/datasets_90_csv\\998febc9d9d5b24f028cbc56bd7b26b9bd37fc3b23de509574150cff61e0c900.text.csv': '62acddae-6081-4f24-b3ad-b69877c0a8d9', 'benchmark/datasets_90_csv\\272911234aba536410e9fb0d0ed5830b10c56a1e6edc3f36600a49c380c12df6.text.csv': '8118cae0-9b48-45b3-98d5-e7da43dfbd4c', 'benchmark/datasets_90_csv\\fdecb95234b4335cd1f7df01f525389dc1b08670a1c12a04a8aa38ed6dabc1cd.text.csv': '8f0c5c2c-9bf1-4

In [11]:
import pickle

with open('dataset_info_benchmark.pkl', 'rb') as f:
    dataset_info = pickle.load(f)

print(dataset_info)

[{'dataset_name': 'benchmark/datasets_90_csv\\02b8096e28ba3e68827c3c1ec3010e497e95dbf477d4ba7cc7cdc9292080b582.text.csv', 'data_profile': 'The key data profile information for the dataset 02b8096e28ba3e68827c3c1ec3010e497e95dbf477d4ba7cc7cdc9292080b582.text includes:\n**Year**: Data is of type text. There are 10 unique values. This column is numeric. Mean: 2005.5, Max: 2010, Min: 2001. \n**Emergency Department Visits**: Data is of type float. This column is numeric. Mean: 521.5, Max: 715.7, Min: 420.6. Coverage spans from 0 to 715.7. \n**Hospitalizations**: Data is of type float. This column is numeric. Mean: 92.89000000000001, Max: 98.7, Min: 82.7. Coverage spans from 0 to 98.7. \n**Deaths**: Data is of type float. This column is numeric. Mean: 18.009999999999998, Max: 18.6, Min: 17.1. Coverage spans from 0 to 18.6. \n**Total**: Data is of type float. This column is numeric. Mean: 631.6899999999998, Max: 823.7, Min: 521.0. Coverage spans from 0 to 823.7. ', 'semantic_profile': '{\n  "

In [13]:
# Crea una struttura dati: {query_id: [{'dataset_id': ..., 'data_profile': ..., 'semantic_profile': ...}, ...]}
query_datasets_info = {}

for query_id, ds_map in dataset_ids_from_jsonl.items():
    query_datasets_info[query_id] = []
    for ds_path, dataset_id in ds_map.items():
        # Find the dictionary in dataset_info where 'dataset_name' matches ds_path
        info = next((d for d in dataset_info if d['dataset_name'] == ds_path), {})
        query_datasets_info[query_id].append({
            'dataset_id': dataset_id,
            'data_profile': info.get('data_profile', ''),
            'semantic_profile': info.get('semantic_profile', '')
        })

print(query_datasets_info)

{'DS1-E-0005': [{'dataset_id': '0cbb09dc-4012-4f18-900e-6d746bfc10d3', 'data_profile': 'The key data profile information for the dataset 4583e0c59aba1daa2b75df1559e23dc0f12048847577f17d482690e37eba415d.text includes:\n**Year and Quarter**: Data is of type text. There are 12 unique values. \n**Topic**: Data is of type text. There are 2 unique values. Top 3 frequent values: Rates by Cause, Rates by Age. \n**Indicator**: Data is of type text. There are 8 unique values. \n**Time Period**: Data is of type text. There are 1 unique values. \n**Rate**: Data is of type float. This column is numeric. Mean: 38.54739583333333, Max: 111.98, Min: 1.86. Coverage spans from 0 to 111.62. \n**Unit**: Data is of type text. There are 2 unique values. Top 3 frequent values: per 100,000 births, per 1,000 births. \n**Significant**: Data is of type text. There are 1 unique values. \n**Standard Error**: Data is of type missingdata. This column is numeric. Mean: nan, Max: nan, Min: nan. \n**Footnote Symbol**: D

Ora, dopo aver estratto i risultati migliori abbiamo bisogno che un reranker analizzi il tutto e rimetta in ordine i risultati

In [16]:
def build_prompts_for_all_queries(query_datasets_info):
    prompts = {}
    for query_id, datasets in query_datasets_info.items():
        prompt = f"""
        Rank the following datasets based on their relevance to the query: "{query_id}".
        The output format is: {query_id} Q0 [dataset_id] [Rank] [Score] MySystem_RAG
        Select a score from 0 to 1, where 0 means not relevant and 1 means very relevant, please use up to 3 decimal digits.
        No additional text is needed, just the output in the specified format.

        The datasets are:
        """
        for ds in datasets:
            prompt += f"""
        Dataset: {ds['dataset_id']}
        --------------------
        Semantic Profile:
        {ds['semantic_profile']}

        Data Profile:
        {ds['data_profile']}
        --------------------
        """
        prompts[query_id] = prompt
    return prompts

In [17]:
build_prompts_for_all_queries(query_datasets_info)

{'DS1-E-0005': '\n        Rank the following datasets based on their relevance to the query: "DS1-E-0005".\n        The output format is: DS1-E-0005 Q0 [dataset_id] [Rank] [Score] MySystem_RAG\n        Select a score from 0 to 1, where 0 means not relevant and 1 means very relevant, please use up to 3 decimal digits.\n        No additional text is needed, just the output in the specified format.\n\n        The datasets are:\n        \n        Dataset: 0cbb09dc-4012-4f18-900e-6d746bfc10d3\n        --------------------\n        Semantic Profile:\n        {\n  "Temporal": {\n    "isTemporal": true,\n    "resolution": "Year and Quarter"\n  },\n  "Spatial": {\n    "isSpatial": false,\n    "resolution": ""\n  },\n  "Entity Type": "Health Indicator",\n  "Domain-Specific Types": "Healthcare",\n  "Function/Usage Context": "Measurement and Aggregation Key"\n}\n\n        Data Profile:\n        The key data profile information for the dataset 4583e0c59aba1daa2b75df1559e23dc0f12048847577f17d482690e

In [18]:
prompts = build_prompts_for_all_queries(query_datasets_info)

with open("results.txt", "w", encoding="utf-8") as f:
    for query_id, prompt in prompts.items():
        reranking = utils.call_openai_api(prompt, "o1-mini")
        output = reranking.choices[0].message.content
        f.write(f"Query ID: {query_id}\n")
        f.write(output + "\n\n")

In [19]:
# Estrai tutte le coppie (query_id, dataset_id) da result.txt
result_pairs = set()
with open("results.txt", "r", encoding="utf-8") as f:
    for line in f:
        # Cerca righe che corrispondono al formato: [query_id] Q0 [dataset_id] ...
        parts = line.strip().split()
        if len(parts) >= 4 and parts[1] == "Q0":
            query_id = parts[0]
            dataset_id = parts[2]
            result_pairs.add((query_id, dataset_id))

# Filtra le righe dal file qrels mantenendo solo quelle con (query_id, dataset_id) in result_pairs
filtered_lines = []
with open(r"benchmark\data_search_2_e_train_qrels.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 3:
            query_id = parts[0]
            dataset_id = parts[1]
            if (query_id, dataset_id) in result_pairs:
                filtered_lines.append(line.strip())

# Scrivi il risultato su un nuovo file
with open("qrels_filtered_by_result.txt", "w", encoding="utf-8") as f:
    for line in filtered_lines:
        f.write(line + "\n")

In [89]:
%pip install -q trectools

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
from trectools import TrecRun, TrecQrel, TrecEval

# Carica il file di run e il file di qrels
run = TrecRun("results.txt")
qrels = TrecQrel("qrels_filtered_by_result.txt")

results = TrecEval(run, qrels)

# Stampa alcune metriche comuni
print(f"nDCG@10: {results.get_ndcg(depth=10)}")
print(f"nDCG@5: {results.get_ndcg(depth=5)}")
print(f"nDCG@3: {results.get_ndcg(depth=3)}")
print(f"Precision@10: {results.get_precision(depth=10)}")
print(f"Recall: {results.get_recall()}")

nDCG@10: 0.6792208846448656
nDCG@5: 0.5903833735657601
nDCG@3: 0.526765225466635
Precision@10: 0.5599999999999999
Recall: 0.8


In [ ]:
print(f"MRR: {results.get_reciprocal_rank()}")

MRR: 0.7


C:\Users\ricca\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\trectools\trec_eval.py:311: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  selection = selection[~selection["rel"].isnull()].groupby("query").first().copy()


In [ ]:
import pytrec_eval

# Le metriche nERR e Q-measure non sono direttamente disponibili in trectools.
# Puoi calcolarle manualmente oppure utilizzare la libreria pytrec_eval se disponibile.
# Esempio di utilizzo con pytrec_eval (se installata):


# Prepara i dizionari per pytrec_eval
qrel_dict = qrels.qrels_data.set_index(['query', 'docid'])['rel'].unstack(fill_value=0).T.to_dict()
run_dict = run.run_data.groupby('query').apply(
    lambda x: dict(zip(x['docid'], x['score']))
).to_dict()

evaluator = pytrec_eval.RelevanceEvaluator(qrel_dict, {'err', 'q_measure'})

metrics = evaluator.evaluate(run_dict)

# Calcola la media delle metriche su tutte le query
mean_nerr_5 = sum([v['err_5'] for v in metrics.values()]) / len(metrics)
mean_nerr_10 = sum([v['err_10'] for v in metrics.values()]) / len(metrics)
mean_q_5 = sum([v['q_measure_5'] for v in metrics.values()]) / len(metrics)
mean_q_10 = sum([v['q_measure_10'] for v in metrics.values()]) / len(metrics)

print(f"nERR@5 (pytrec_eval): {mean_nerr_5}")
print(f"nERR@10 (pytrec_eval): {mean_nerr_10}")
print(f"Q-measure@5 (pytrec_eval): {mean_q_5}")
print(f"Q-measure@10 (pytrec_eval): {mean_q_10}")


AttributeError: 'TrecEval' object has no attribute 'get_err'